# SIR Infection Simulation

Interactive SIR compartment model with adjustable parameters. Move the sliders below to recompute and replot the epidemic curves.

The model integrates:

- **dS/dt** = −β · S · I / N
- **dI/dt** = β · S · I / N − γ · I
- **dR/dt** = γ · I

with constant population size **N**.

In [3]:
%matplotlib inline

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from sir_model import SIRParameters, run_sir_simulation

In [ ]:
def plot_sir(
    N: float,
    beta: float,
    gamma: float,
    S0: float,
    I0: float,
    R_init: float,
    t_max: float,
) -> None:
    try:
        params = SIRParameters(
            N=N,
            beta=beta,
            gamma=gamma,
            S0=S0,
            I0=I0,
            R0=R_init,
            t_max=t_max,
        )
        result = run_sir_simulation(params)
    except (ValueError, RuntimeError) as exc:
        print(f"Invalid parameters: {exc}")
        return

    S0_norm, I0_norm, R0_norm = params.normalized_initial_conditions()

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(result.t, result.S, color="#2563eb", linewidth=2, label="Susceptible (S)")
    ax.plot(result.t, result.I, color="#dc2626", linewidth=2, label="Infected (I)")
    ax.plot(result.t, result.R, color="#16a34a", linewidth=2, label="Recovered (R)")
    ax.set_xlabel("Time")
    ax.set_ylabel("Population")
    ax.set_title("SIR Compartment Model")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")

    stats = (
        f"ℛ₀ = β/γ ≈ {params.basic_reproduction_number:.2f}\n"
        f"R_eff(0) ≈ {params.effective_reproduction_number:.2f}\n"
        f"Peak infected: {result.peak_infected:.0f} at t = {result.peak_time:.1f}\n"
        f"Initial (normalized): S={S0_norm:.0f}, I={I0_norm:.0f}, R={R0_norm:.0f}\n"
        f"Final: S={result.S[-1]:.0f}, I={result.I[-1]:.0f}, R={result.R[-1]:.0f}"
    )
    ax.text(
        0.02,
        0.98,
        stats,
        transform=ax.transAxes,
        verticalalignment="top",
        fontsize=10,
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85},
    )
    plt.show()


controls = widgets.interactive(
    plot_sir,
    N=widgets.IntSlider(min=100, max=10000, step=100, value=1000, description="N"),
    beta=widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.3, description="β"),
    gamma=widgets.FloatSlider(min=0.01, max=1.0, step=0.01, value=0.1, description="γ"),
    S0=widgets.IntSlider(min=0, max=10000, step=10, value=990, description="S₀"),
    I0=widgets.IntSlider(min=0, max=1000, step=1, value=10, description="I₀"),
    R_init=widgets.IntSlider(min=0, max=1000, step=1, value=0, description="R(0)"),
    t_max=widgets.FloatSlider(min=20, max=500, step=10, value=160, description="t max"),
)

controls.layout = widgets.Layout(width="500px")
display(controls)

interactive(children=(IntSlider(value=1000, description='N', max=10000, min=100, step=100), FloatSlider(value=…